In [1]:
import os
import json
import sys

TASK3_DIR = os.path.abspath("../task3")
if TASK3_DIR not in sys.path:
    sys.path.append(TASK3_DIR)

from rag_module import NovaRAG

c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rag = NovaRAG()
rag.ingest_documents()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1290.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2262.18it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'ingested_count': 11, 'total_known_docs': 11}

In [3]:
result = rag.answer_query("Is this serum good for oily skin?")
print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for i, src in enumerate(result["sources"], 1):
    print(f"\nSource {i}:")
    print("ID:", src["id"])
    print("Metadata:", src["metadata"])
    print("Rerank Score:", src.get("rerank_score"))
    print("Text:", src["text"][:300])

ANSWER:
Based on the knowledge base, this appears suitable for oily skin. The answer is grounded in the product target and ingredient details retrieved from the catalog.

SOURCES:

Source 1:
ID: product_P001
Metadata: {'product_id': 'P001', 'category': 'skincare', 'source': 'product_catalog', 'name': 'Hydrating Face Serum'}
Rerank Score: -1.1988184452056885
Text: Product ID: P001
Product Name: Hydrating Face Serum
Category: skincare
Ingredients: Hyaluronic Acid, Vitamin E
Suitable For: dry,sensitive
Price: 40
Description: Hydrating Face Serum is a skincare product. It contains Hyaluronic Acid, Vitamin E and is intended for dry,sensitive.

Source 2:
ID: faq_ingredients_1
Metadata: {'source': 'faq', 'topic': 'ingredients'}
Rerank Score: -1.7101281881332397
Text: FAQ: Customers often ask whether skincare products are suitable for oily, dry, or sensitive skin. Product suitability should be based on the product target field and listed ingredients.

Source 3:
ID: faq_recommendation_1
Metadat

In [4]:
test_queries = [
    "Is this serum good for oily skin?",
    "Suggest something for dry skin",
    "Can I return a damaged product?",
    "How do I know my jacket size?",
    "Does this product contain salicylic acid?"
]

for q in test_queries:
    result = rag.answer_query(q)
    print("\n" + "=" * 100)
    print("QUERY:", q)
    print("ANSWER:", result["answer"])
    print("TOP SOURCES:", [s["id"] for s in result["sources"]])


QUERY: Is this serum good for oily skin?
ANSWER: Based on the knowledge base, this appears suitable for oily skin. The answer is grounded in the product target and ingredient details retrieved from the catalog.
TOP SOURCES: ['product_P001', 'faq_ingredients_1', 'faq_recommendation_1']

QUERY: Suggest something for dry skin
ANSWER: Based on the knowledge base, hydrating products targeted for dry skin are the most relevant match.
TOP SOURCES: ['faq_recommendation_1', 'faq_ingredients_1', 'product_P001']

QUERY: Can I return a damaged product?
ANSWER: According to NOVA policy, returns can be requested for eligible products within 7 days of delivery, and the order ID is required to initiate the process.
TOP SOURCES: ['faq_returns_1', 'policy_returns_1', 'policy_safety_1']

QUERY: How do I know my jacket size?
ANSWER: According to NOVA sizing guidance, customers should compare their usual fit with available sizes like S, M, L, and XL because fit can vary by apparel product.
TOP SOURCES: ['

In [5]:
eval_cases = [
    {
        "query": "Can I return a damaged product?",
        "expected_keyword": "returns"
    },
    {
        "query": "Is this serum good for oily skin?",
        "expected_keyword": "oily skin"
    },
    {
        "query": "What sizes are available for apparel?",
        "expected_keyword": "sizes"
    },
    {
        "query": "What should I use for dry skin?",
        "expected_keyword": "dry skin"
    }
]

evaluation = rag.evaluate(eval_cases)
print(json.dumps(evaluation, indent=2))

{
  "num_cases": 4,
  "accuracy": 1.0,
  "results": [
    {
      "query": "Can I return a damaged product?",
      "expected_keyword": "returns",
      "answer": "According to NOVA policy, returns can be requested for eligible products within 7 days of delivery, and the order ID is required to initiate the process.",
      "hit": true,
      "num_sources": 3
    },
    {
      "query": "Is this serum good for oily skin?",
      "expected_keyword": "oily skin",
      "answer": "Based on the knowledge base, this appears suitable for oily skin. The answer is grounded in the product target and ingredient details retrieved from the catalog.",
      "hit": true,
      "num_sources": 3
    },
    {
      "query": "What sizes are available for apparel?",
      "expected_keyword": "sizes",
      "answer": "According to NOVA sizing guidance, customers should compare their usual fit with available sizes like S, M, L, and XL because fit can vary by apparel product.",
      "hit": true,
      "num

In [6]:
output_path = os.path.join("../task3", "evaluation_report.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(evaluation, f, indent=2)

print("Saved:", output_path)

Saved: ../task3\evaluation_report.json
